In [1]:
import os
import re
import fnmatch
import glob
import sys
module_path = os.path.abspath(os.path.join('./src/python'))
sys.path.append(module_path)
import afim
import xesmf             as xe
import numpy             as np
import pandas            as pd
import xarray            as xr
import cosima_cookbook   as cc
import matplotlib.pyplot as plt
import cartopy.crs       as ccrs
import cmocean
from dask.distributed import Client
from datetime         import datetime
import importlib
%matplotlib inline

In [8]:
# this only needs to be run if you've made changes to AFIM after loading it the first time
importlib.reload(afim)

<module 'afim' from '/home/581/da1339/src/python/afim.py'>

# the complete method
## this next cell will take roughly 2.5 days (real-time) per dataset-year

In [9]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
cice_prep.acom2_compute_qdp()

extracting ac-om2 data from model run:  01deg_jra55v140_iaf  from start dt:  2005-01-01 12:00  to stop dt:  2005-12-31 12:00
computing temperature derivative over time
generating regridding weights on the fly using xesmf via method:  bilinear

source grid, looks like this:  <xarray.Dataset>
Dimensions:  (ny: 721, nx: 1440)
Coordinates:
    lon      (ny, nx) float32 ...
    lat      (ny, nx) float32 ...
Dimensions without coordinates: ny, nx
Data variables:
    *empty*
Attributes:
    NCO:      netCDF Operators version 5.1.0 (Homepage = http://nco.sf.net, C...
    history:  Fri Sep  9 12:19:06 2022: ncatted -a units,lon,c,c,degrees east...
/g/data/jk72/da1339/grids/weights/map_ERA5_access-om2_cice_0p1_bilinear.nc
generating regridding weights on the fly using xesmf via method:  bilinear

source grid, looks like this:  <xarray.Dataset>
Dimensions:  (ny: 721, nx: 1440)
Coordinates:
    lon      (ny, nx) float32 ...
    lat      (ny, nx) float32 ...
Dimensions without coordinates: ny, nx
D

KeyboardInterrupt: 

# Below is for troubleshooting the the big method above

## request AC-OM2-01 fields 

In [ ]:
session = cc.database.create_session()
temp = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='temp',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
salt = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='salt',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
mld  = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='mld',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')

# compute net atmospheric surface heat flux

In [ ]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
G_CICE    = cice_prep.cice_grid_preperation()
G_ERA5    = cice_prep.era5_grid_prep()
yr_str    = str(datetime.strptime(cice_prep.start_date,'%Y-%m-%d').year)
rg        = xe.Regridder(G_ERA5, G_CICE, method='bilinear', periodic=True, filename=cice_prep.F_ERA5_weights, reuse_weights=True)
lw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwlwrf',yr_str,'*.nc') , chunks={'lat':100,'lon':100} )
sw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwswrf',yr_str,'*.nc') , chunks={'lat':100,'lon':100} ) 
msshf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msshf',yr_str,'*.nc') , chunks={'lat':100,'lon':100} ) 
mslhf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'mslhf',yr_str,'*.nc') , chunks={'lat':100,'lon':100} )
F_net_nat = sw_nat.msdwswrf - lw_nat.msdwlwrf - mslhf_nat.mslhf - msshf_nat.msshf
F_net_rg  = rg(F_net_nat)
F_net_rg  = F_net_rg.assign_coords(lon=G_CICE.lon[0,:],lat=G_CICE.lat[:,0])
F_net_rg  = F_net_rg.rename({'y':'yt_ocean','x':'xt_ocean'})
F_net_rg  = F_net_rg.rename({'lat':'yt_ocean','lon':'xt_ocean'})
F_net_rg  = F_net_rg.resample(time='1D').mean()

# compute temperature time derivative

requires re-chunking of time dimension, which is currently chunked to 1, to be greater than 2,

In [ ]:
temp = temp.chunk({'time':3})
dTdt = temp.differentiate("time")
dTdt = dTdt.isel(time=0).sel(st_ocean=mld,method='nearest').drop('st_ocean')

# slice temperature and salt at the mixed layer depth

In [ ]:
T_mld = temp.isel(time=0).sel(st_ocean=mld.isel(time=0),method='nearest')
S_mld = salt.isel(time=0).sel(st_ocean=mld.isel(time=0),method='nearest')
#T_mld.plot(figsize=(20,10))
#S_mld.plot(figsize=(20,10))

# compute heat capicity and water density at the mixed layer depth

In [ ]:
cp  = afim.compute_ocn_heat_capacity_at_depth(S_mld, T_mld-273.15, mld.isel(time=0), mld.yt_ocean)
rho = afim.compute_ocn_density_at_depth(S_mld, T_mld-273.15, mld.isel(time=0), mld.yt_ocean)
#cp.plot(figsize=(20,10))
#rho.plot(figsize=(20,10))

# compute ocean heat flux at the mixed layer depth

In [ ]:
qdp = afim.compute_ocn_heat_flux_at_depth(rho,cp,mld.isel(time=0),F_net_rg.isel(time=0),dTdt)
qdp

In [ ]:
qdp.isel(time=180).plot(figsize=(20,10))